# Notebook 06 - Coupled Piping-Structural Systems and Support Optimization

This lesson extends the pipe-only workflow into a coupled pipe-rack model. It loads Code_Aster-backed results for scoring, then shows how iterative support optimization can be enabled when a configured solver runtime is available.

You will do four things:

1. Build a portal frame and pipe crossing in one `TubaModel`.
2. Add a frictional rest support at the actual crossing node.
3. Score the design from Code_Aster-backed results.
4. Keep iterative optimization explicitly gated behind a solver-runtime flag.


## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pyvista as pv

# Setup repo root for path import
REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tuba import Model
from tuba.analysis.code_aster_notebook import configure_code_aster_notebook_runtime, load_or_run_code_aster_results
from tuba.visualizer import plots

# Enable interactive notebook rendering
# Defaults to zoomable embedded HTML locally; set TUBA_NOTEBOOK_BACKEND=client or static to override.
from tuba.visualizer.notebook import configure_notebook_backend
JUPYTER_BACKEND = configure_notebook_backend()

## 2. Building the Coupled Portal Frame + Piping Model

We'll build a structural portal frame at $X = 5.0$ meters:
- Two vertical columns modeled as beams using a box section profile (`ColBox`).
- One horizontal girder connecting the column tops modeled as an I-beam (`HE200B`).
- A piping system crossing directly over the girder at $Y = 3.1143$ meters with a frictional rest support.

In [ ]:
model = Model("StructuralDemo", standard="ASME_B31.3")

model.add_material(
    "S235JR",
    E=2.1e11,
    nu=0.3,
    alpha=1.2e-5,
    rho=7850.0,
    allowable_stress={20.0: 137e6, 200.0: 120e6},
)
model.add_pipe_section("4inch_sch40", OD=0.1143, WT=0.00602)
model.add_rectangular_section("ColBox", height_y=0.2, height_z=0.2, thickness_y=0.008, thickness_z=0.008)
model.add_ibeam_section("HE200B_Girder", "HE200B")

c1_base = model.add_node([5.0, 0.0, -1.0])
c1_top = model.add_node([5.0, 3.0, -1.0])
model.add_element(id="col_1", type="beam", n1=c1_base, n2=c1_top, section="ColBox", material="S235JR")
model.add_support(c1_base, "anchor")

c2_base = model.add_node([5.0, 0.0, 1.0])
c2_top = model.add_node([5.0, 3.0, 1.0])
model.add_element(id="col_2", type="beam", n1=c2_base, n2=c2_top, section="ColBox", material="S235JR")
model.add_support(c2_base, "anchor")

model.add_element(id="girder", type="beam", n1=c1_top, n2=c2_top, section="HE200B_Girder", material="S235JR")

with model.pipe(section="4inch_sch40", material="S235JR") as b:
    b.start([0, 3.1143, 0], support="anchor")
    b.run(5.0)
    b.run(5.0)
    b.end(support="anchor")

pipe_crossing_node = next(
    nid for nid, node in model.nodes.items()
    if np.allclose(node.coords, [5.0, 3.1143, 0.0])
)
print(f"Crossing node over the girder: {pipe_crossing_node}")
print(f"Model has {len(model.nodes)} nodes and {len(model.elements)} elements.")


Attach the frictional rest to the computed crossing node directly above the structural girder.


In [ ]:
model.add_support(pipe_crossing_node, type="rest", friction_coefficient=0.3)
model.define_load_case("Operating_Hot", gravity=True, pressure=2.5e6, temperature=220.0, ref_temperature=20.0)
print(f"Rest support attached to {pipe_crossing_node}.")


### Visualizing the Coupled Structural-Piping Rack

In [ ]:
from tuba.visualizer.pipeline import build_mesh_from_model, inflate_tubes

mesh = build_mesh_from_model(model)
tubes = inflate_tubes(mesh, radius=0.05)

p = pv.Plotter()
p.set_background("#1a1a2e")
p.add_mesh(tubes, color="#5c6b73")
plots._add_supports_to_plotter(p, model, scale=0.18)
p.show(jupyter_backend=JUPYTER_BACKEND)

## 3. Multi-Objective Design Evaluation

Engineering optimization requires balancing multiple competing objectives.
Tuba provides an `ObjectiveEvaluator` class that aggregates design penalties for:
1. **Stress:** Violations of ASME B31.3 allowable limits.
2. **Deflection:** Sag displacements exceeding the 2.5 mm limit.
3. **Support Cost:** Support counts and complexity metrics.
4. **Clash:** Interference checks against obstacles.

In [ ]:
from tuba.routing.objectives import (
    ObjectiveEvaluator,
    StressObjective,
    DeflectionObjective,
    SupportCostObjective,
    ClashObjective,
)

# Create the evaluator
evaluator = ObjectiveEvaluator([
    StressObjective(weight=1.0),
    DeflectionObjective(weight=2.0),
    SupportCostObjective(weight=0.5),
    ClashObjective(weight=3.0),
])

CODE_ASTER_RUNTIME = configure_code_aster_notebook_runtime()
RUN_CODE_ASTER = True
CODE_ASTER_WORK_DIR = REPO_ROOT / "notebooks" / "code_aster_results" / "structural_operating_hot"

code_aster_run = load_or_run_code_aster_results(
    model,
    "Operating_Hot",
    CODE_ASTER_WORK_DIR,
    run_solver=RUN_CODE_ASTER,
    exec_method=CODE_ASTER_RUNTIME.exec_method,
    wsl_distro=CODE_ASTER_RUNTIME.wsl_distro,
    docker_image=CODE_ASTER_RUNTIME.docker_image,
)
results = code_aster_run.results
code_aster_artifact = code_aster_run.artifact

if code_aster_run.ran_solver:
    print("Code_Aster solver executed for this notebook run.")
scores = evaluator.get_detailed_scores(model, results)
print("Multi-objective score breakdown from Code_Aster results:")
for obj_name, info in scores.items():
    if isinstance(info, dict):
        print(f"  {obj_name}: Penalty = {info['score']:.3f}, Details = {info['details']}")
    else:
        print(f"  {obj_name}: {info:.3f}")

## 4. Optional Iterative Support Optimization

`RuleBasedSupportPlacer` can iterate support layouts and dispatch Code_Aster for every candidate. Keep this disabled until the configured runtime is available and you intentionally want repeated solves.


In [ ]:
from tuba.routing.optimizer import RuleBasedSupportPlacer

RUN_SOLVER_OPTIMIZATION = False
opt_model = model
opt_res = results

if RUN_SOLVER_OPTIMIZATION:
    placer = RuleBasedSupportPlacer(solver_name="code_aster", deflection_limit_m=0.0025)
    print(f"Supports count before optimization: {len(model.supports)}")
    opt_model, opt_res = placer.optimize(model, evaluator)
    if opt_res is None:
        raise RuntimeError("Code_Aster optimization did not return solver results.")
    print(f"Supports count after optimization: {len(opt_model.supports)}")
    for i, sup in enumerate(opt_model.supports):
        print(f"  Support {i}: Node {sup.node}, Type {sup.type}")
else:
    print("Set RUN_SOLVER_OPTIMIZATION = True after the configured Code_Aster runtime is available for iterative solves.")
    print("The currently loaded Code_Aster result set remains available as opt_res for downstream cells.")

## Key Takeaways

- Mixed structural and piping elements can live in one `TubaModel`.
- The scoring cell uses Code_Aster-backed results loaded through the shared notebook helper.
- Iterative optimization is explicitly gated because it runs repeated solver jobs.

Next: `07_bim_data_exchange.ipynb` exports the model and solver properties into data-exchange formats.
